# Project: kernel showdown
**Machine Learning · Week 4 (Kernels and support vector machines) · Day 5**

Lesson page: https://sara-aminpour.com/tutorials/ml/w4d5.html

In [ ]:
# On Google Colab, this installs qiskit (about a minute). On your own computer, skip it and
# install the packages once instead: pip install qiskit==2.5.2 scikit-learn torch matplotlib
import sys
if "google.colab" in sys.modules:
    %pip install --quiet qiskit==2.5.2

Project: a kernel showdown. Four datasets, four kernels, one fair contest: hyperparameters chosen by
cross-validation on the training set, and the test set used once at the end.

One of the kernels is special. Encode each feature $x_j$ as a rotation $R_Y(s\,x_j)$ of its own qubit.
The overlap of two such states is
$$k(x, x') = |\langle\varphi(x)|\varphi(x')\rangle|^2 = \prod_j \cos^2\!\Big(\frac{s\,(x_j - x'_j)}{2}\Big)$$
a **quantum kernel** that is simple enough to compute on a laptop.

In [ ]:
import numpy as np
from qiskit import QuantumCircuit
from qiskit.quantum_info import Statevector

def encode(x, s=1.0):
    """One qubit per feature, rotated by RY(s * x_j)."""
    qc = QuantumCircuit(len(x))
    for j, v in enumerate(x):
        qc.ry(s * v, j)
    return Statevector(qc)

x, z = np.array([0.3, -1.2]), np.array([1.1, 0.4])
quantum = abs(encode(x).inner(encode(z))) ** 2
formula = np.prod(np.cos((x - z) / 2) ** 2)
print(f"overlap of the two quantum states: {quantum:.6f}")
print(f"product-of-cosines formula:        {formula:.6f}")

## The contestants

In [ ]:
from sklearn.datasets import make_moons, make_circles, make_blobs
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC

rng = np.random.default_rng(0)
Xx = rng.uniform(-1, 1, size=(300, 2))
datasets = {
    "moons": make_moons(300, noise=0.2, random_state=1),
    "circles": make_circles(300, noise=0.1, factor=0.45, random_state=1),
    "xor": (Xx, (Xx[:, 0] * Xx[:, 1] > 0).astype(int)),
    "blobs": make_blobs(300, centers=[(-1, -0.6), (1, 0.6)], cluster_std=0.6, random_state=1),
}
print({name: X.shape for name, (X, y) in datasets.items()})

## Project: run the showdown
1. Write `angle_kernel(A, B, s)`: the matrix of $\prod_j \cos^2(s(a_j - b_j)/2)$ for every row of A and B.
2. For each dataset: standardize, split, and for each kernel let `GridSearchCV` choose C (and the kernel's knob)
   with 5-fold cross-validation. A callable kernel can be passed straight to `SVC(kernel=...)`.
3. Print a table of **test** accuracies.

In [ ]:
def angle_kernel(A, B, s=1.0):
    return np.zeros((len(A), len(B)))     # TODO: broadcast A[:, None, :] - B[None, :, :], then cos^2 and a product over features

grids = {
    "linear": (SVC(kernel="linear"), {"C": [0.1, 1, 10]}),
    "poly": (SVC(kernel="poly", degree=3, coef0=1), {"C": [0.1, 1, 10]}),
    "rbf": (SVC(kernel="rbf"), {"C": [0.1, 1, 10], "gamma": [0.1, 1, 10]}),
}
print("angle kernel shape check:", angle_kernel(np.zeros((3, 2)), np.zeros((4, 2))).shape)

---
## Solution
Try the exercise yourself first: it's fine to get stuck for a minute. Then compare with this one (there are many good answers).

In [ ]:
def angle_kernel(A, B, s=1.0):
    d = A[:, None, :] - B[None, :, :]                      # (len(A), len(B), features)
    return np.prod(np.cos(s * d / 2) ** 2, axis=-1)

print(f"{'dataset':>8} | {'linear':>7} {'poly':>7} {'rbf':>7} {'quantum':>8}")
for name, (X, y) in datasets.items():
    X = StandardScaler().fit_transform(X)
    X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.3, random_state=0, stratify=y)
    row = []
    for kname in ["linear", "poly", "rbf"]:
        model, grid = grids[kname]
        search = GridSearchCV(model, grid, cv=5).fit(X_tr, y_tr)
        row.append(search.score(X_te, y_te))
    # the quantum kernel's knob is the scale s: choose it by cross-validation too
    best_search, best_s = None, None
    for s in [0.5, 1.0, 2.0, 3.0]:
        kernel = lambda A, B, s=s: angle_kernel(A, B, s)
        search = GridSearchCV(SVC(kernel=kernel), {"C": [0.1, 1, 10]}, cv=5).fit(X_tr, y_tr)
        if best_search is None or search.best_score_ > best_search.best_score_:
            best_search, best_s = search, s
    row.append(best_search.score(X_te, y_te))
    print(f"{name:>8} | " + " ".join(f"{a:7.2f}" for a in row[:3]) + f" {row[3]:8.2f}   (quantum scale s = {best_s})")
print("\nThe linear kernel loses wherever the boundary curves, and this simple quantum kernel never beats a well-tuned")
print("RBF: for small differences cos^2(d/2) is close to exp(-d^2/4), so it is a similar kind of similarity.")
print("Finding data where a quantum kernel really helps is an open research question.")